# **Note**: To see the details that main_notebook.ipynb didn't talk about jump right to 2.3.1 - 2.3.3

### Are parking prices related to how busy an area is?

### 2.1. Why this question is valuable
Understanding whether **parking prices reflect how busy an area is** helps evaluate if pricing is working as a demand-management tool. 

If parking prices **match** how busy an area is, then:

- **Higher demand → higher price → fewer people choose to park there → more turnover → better chance of finding a space.**
- **Lower demand → lower price → attracts drivers to underused areas → balances parking usage across the city.**
- **Prices aligned with demand → revenue reflects usage → more funding for maintenance and transport.**

If prices **don’t** match busyness, then:

- **Busy area + low price → too many drivers → full spaces → people circle longer → more congestion.**
- **Quiet area + high price → few drivers → empty spaces → wasted supply and lower economic activity nearby.**
- **Mismatch → pricing feels unfair and policy becomes less effective.**

### 2.2. Method
We treat this as a **relationship analysis** between two zone-level variables: **price** and **busyness**. 

To compute these, we first construct our two zone-level variables from the parking tables:

- **Define the analysis unit (zones)**  
   Use **parking_zones.csv** to identify and label zones, and use the zone identifiers needed to join across datasets.

- **Compute zone-level price**  
   Use **parking_rates.csv** to get meter hourly rates, then aggregate within each zone (**mean rate**) to represent the typical price level.

- **Compute zone-level busyness (transactions per space)**  
   - Sum total transactions per zone from **payment_transactions.csv**.  
   - Get total parking capacity per zone from **parking_spaces_zone_group_norm.csv**.  
   - Compute the busyness proxy (zone-level):  
   $$
   \text{transactions\_per\_space}=\frac{\text{total\_transactions}}{\text{total\_spaces}}
   $$

Once we have these zone-level variables, we analyze how they relate:

- **Compare price vs. busyness (visual check)**  
  Create bar charts that rank zones by **mean rate** and by **transactions\_per\_space**, then compare the patterns side-by-side.  
  If pricing reflects demand, zones that appear near the top of the busyness ranking should also tend to appear near the top of the price ranking.

- **Diagnose mismatches (actionable cases)**  
  Identify zones that fall into “unexpected” categories such as **busy-but-cheap** (high busyness, low price) or **quiet-but-expensive** (low busyness, high price).  
  These mismatched zones are treated as candidates for further policy review—potentially involving pricing updates, enforcement-hour changes, or adjustments to supply constraints.

### 2.3. Assumptions and limitations

We use **transactions per space** as a proxy for **busyness**, not a direct measure of real-time occupancy. A higher value suggests a zone is more active, but it may also reflect faster turnover rather than spaces being continuously full.

Both **price** and **busyness** are measured at the **zone level**. This simplifies comparison, but it can hide important variation within a zone, since some meters or blocks may be much busier than others.

For **price**, we use the **mean hourly rate** within each zone. This gives a representative zone-level price, but it may hide more complex pricing rules such as different meter schedules, time-based rates, or special cases within the same zone.

#### 2.4. Code

**Imports**

In [1]:
import polars as pl
import altair as alt
# ----------------
# DATA IMPORTS
# ----------------
parking_rates = pl.read_csv("../final_datasets/parking_rates.csv")
parking_meters = pl.read_csv("../final_datasets/parking_meters.csv")
# parking spaces (zone level)
parking_spaces_zone_group_norm = pl.read_csv("../final_datasets/parking_spaces_zone_group_norm.csv")
# payment_transactions (zone level)
payment_transactions = pl.read_csv("../final_datasets/payment_transactions.csv")
# used to join zone-level results to other parking tables keyed by zone_id
# i.e. parking_xxx.csv 
zone_group_norm_to_zone_id = pl.read_csv("../final_datasets/zone_group_norm_to_zone_id.csv")

# define joinable zone list
joinable_zones_lst = (
    parking_rates.join(zone_group_norm_to_zone_id, on="zone_id")
    # we have one to many zone_group_norm - zone_id mapping. See datasets_join.ipynb for details
    .unique(subset="zone_group_norm",keep="first")
    .join(parking_spaces_zone_group_norm, on="zone_group_norm")
    .join(payment_transactions, on="zone_group_norm")
    # some zones cannot be normalized to allow joining between different datasets
    # so they are not joinable. See datasets_join.ipynb for details
    .select("zone_group_norm")
    .unique().to_series().to_list()
)


**Variable Praparation**

In [2]:
# ------------------------------------------------------------
# Keep only zones that can be joined across all required datasets
# ------------------------------------------------------------
rates = (
    parking_rates
    # Remove repeated pricing rules that appear across multiple meters
    .unique(subset=["zone_id", "rate_window", "rate_time", "rate"], keep="first")
    .join(zone_group_norm_to_zone_id, on="zone_id")
    .filter(pl.col("zone_group_norm").is_in(joinable_zones_lst))
)

tx = payment_transactions.filter(
    pl.col("zone_group_norm").is_in(joinable_zones_lst)
)

spaces = parking_spaces_zone_group_norm.filter(
    pl.col("zone_group_norm").is_in(joinable_zones_lst)
)

#### 2.4.1 Rate unique value analysis

In [3]:
# print rate_time unique values
print(rates.select("rate_time").unique().to_series().to_list())

['~ 8AM - 2PM', 'all time', '~ 2PM - 6PM']


Only Oakland 3 has special rate_time

In [4]:
rates.filter(
    (pl.col("rate_time") == "~ 2PM - 6PM")
    | (pl.col("rate_time") == "~ 8AM - 2PM")
).select("zone_group_norm")

zone_group_norm
str
"""OAKLAND 3"""
"""OAKLAND 3"""


In addition, OAKLAND 3 also have rate that is for all time. 

In practice, this means that in Oakland 3 some meters have one constant rate for all times, while others have different prices for different time ranges. See parking_meter_data.ipynb for details of thie raw data preprocessing

In [5]:
rates.filter(
    pl.col("zone_group_norm") == "OAKLAND 3"
)

zone_id,rate_window,rate_time,rate,meter_id,zone_group_norm
str,str,str,f64,str,str
"""OAKLAND3""","""A""","""all time""",3.0,"""OAKLAND3|-79.95131|40.44669""","""OAKLAND 3"""
"""OAKLAND3""","""A""","""~ 8AM - 2PM""",3.0,"""OAKLAND3|-79.94861|40.44588""","""OAKLAND 3"""
"""OAKLAND3""","""B""","""~ 2PM - 6PM""",2.5,"""OAKLAND3|-79.94914|40.44576""","""OAKLAND 3"""


To keep the problem simple, let's just use rate that for is all time for OAKLAND 3

In [6]:
rates = rates.filter(
    ~(
        (pl.col("zone_group_norm") == "OAKLAND 3") &
        (pl.col("rate_time") != "all time")
    )
)
rates.filter(
    pl.col("zone_group_norm") == "OAKLAND 3"
)

zone_id,rate_window,rate_time,rate,meter_id,zone_group_norm
str,str,str,f64,str,str
"""OAKLAND3""","""A""","""all time""",3.0,"""OAKLAND3|-79.95131|40.44669""","""OAKLAND 3"""


#### 2.4.2. Duplicate row cleaning

In [7]:
# zone_group_norm that have multiple rates rows
rates.group_by("zone_group_norm").len().filter(
    pl.col("len") > 1
)

zone_group_norm,len
str,u32
"""SHADYSIDE""",2
"""SQUIRREL HILL""",2
"""HILL DISTRICT""",2
"""UPTOWN""",2
"""OAKLAND 4""",2


This recalls the problem of one to many mapping.

For example UPTOWN1, UPTOWN2 are mapped to UPTOWN

This mapping is required because other datasets just have UPTOWN as zone

See datasets_join.ipynb for details about one to many mapping

In [8]:
# show
dup_list = rates.group_by("zone_group_norm").len().filter(
    pl.col("len") > 1
).select("zone_group_norm").unique().to_series().to_list()
rates.filter(
    pl.col("zone_group_norm").is_in(dup_list)
).sort("zone_group_norm")

zone_id,rate_window,rate_time,rate,meter_id,zone_group_norm
str,str,str,f64,str,str
"""HILL-DIST""","""A""","""all time""",1.0,"""HILL-DIST|-79.98147|40.44289""","""HILL DISTRICT"""
"""HILL-DIST-2""","""A""","""all time""",1.5,"""HILL-DIST-2|-79.96505|40.44517""","""HILL DISTRICT"""
"""OAKLAND4""","""A""","""all time""",3.0,"""OAKLAND4|-79.95399|40.44093""","""OAKLAND 4"""
"""OAKLAND4""","""A""","""all time""",1.5,"""OAKLAND4|-79.94666|40.43984""","""OAKLAND 4"""
"""SHADYSIDE2""","""A""","""all time""",1.5,"""SHADYSIDE2|-79.9419|40.45481""","""SHADYSIDE"""
"""SHADYSIDE1""","""A""","""all time""",1.5,"""SHADYSIDE1|-79.93516|40.45069""","""SHADYSIDE"""
"""SQ.HILL1""","""A""","""all time""",2.0,"""SQ.HILL1|-79.92108|40.43798""","""SQUIRREL HILL"""
"""SQ.HILL2""","""A""","""all time""",2.0,"""SQ.HILL2|-79.92456|40.42839""","""SQUIRREL HILL"""
"""UPTOWN2""","""A""","""all time""",1.5,"""UPTOWN2|-79.98719|40.4386""","""UPTOWN"""


We have different rate for same zone. 

In practice, this means that within a zone there may still be meters with different prices, so we use the average as a summary value for the zone.

Let's take the average of them

In [9]:
avg_rates = (
    rates
    .select(["zone_group_norm", "rate"])
    .group_by("zone_group_norm")
    .mean()
)

#### 2.4.3 Parking Rates Distribution

**Most of the zones apply 1, 1.5 or 2 $/hour**

In [10]:
# Rates percentage distribution
rate_dist = (
    avg_rates
    .group_by("rate")
    .len()
    .rename({"len": "count"})
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).alias("percent")
    )
    .sort("rate")
)
rate_dist_pd = rate_dist.to_pandas()

# donut char bulding
donut = (
    alt.Chart(rate_dist_pd)
    .mark_arc(innerRadius=80)
    .encode(
        theta=alt.Theta("count:Q", stack=True),
        color=alt.Color("rate:N", title="Rate ($/hour)"),
        tooltip=[
            alt.Tooltip("rate:N", title="Rate"),
            alt.Tooltip("count:Q", title="Count"),
            alt.Tooltip("percent:Q", title="Percent", format=".1f"),
        ],
    )
    .properties(
        title="Parking Rate Distribution",
        width=300,
        height=300,
    )
)

# center text (2 lines)
center_line1 = (
    alt.Chart()
    .mark_text(fontSize=10, fontWeight="bold", align="center")
    .encode(text=alt.value("Most zones apply"))
    .properties(width=300, height=300)
)

center_line2 = (
    alt.Chart()
    .mark_text(fontSize=8, align="center", dy=10)
    .encode(text=alt.value("$1, $1.5, or $2 / hour"))
    .properties(width=300, height=300)
)

donut + center_line1 + center_line2

alt.LayerChart(...)

Let's check counts

In [11]:
# Rates distribution (counts)
bars = (
    alt.Chart(rate_dist_pd)
    .mark_bar()
    .encode(
        x=alt.X("rate:O", 
                title="Rate ($/hour)", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("count:Q", title="Number of zones"),
        color = alt.Color("count:Q"),
        tooltip=[
            alt.Tooltip("rate:O", title="Rate"),
            alt.Tooltip("count:Q", title="N_zone"),
        ],
    )
    .properties(
        title="Distribution of Parking Zones by Hourly Rate",
        width=400,
        height=320,
    )
)

bars

alt.Chart(...)

For rate price 1.25 and 2.25 these is only one zone. 

These unusual rate price are due to the avg that we did before.

Let's divide these prices into three categories:
- Low (1, 1.25)
- Medium (1.5, 2, 2,25)
- High (3, 4)

In [12]:
# ------------------------------------------------------------
# Assign each zone to a broad price category, since values like
# 1.25 and 2.25 are derived by averaging rates within a zone
# ------------------------------------------------------------
avg_rates = avg_rates.with_columns(
    pl.when(pl.col("rate").is_in([1.0, 1.25]))
      .then(pl.lit("Group 1: (1$, 1.25$)"))
    .when(pl.col("rate").is_in([1.5]))
      .then(pl.lit("Group 2: (1.5$)"))
    .when(pl.col("rate").is_in([2.0, 2.25]))
      .then(pl.lit("Group 3: (2$, 2.25$)"))
    .when(pl.col("rate").is_in([3.0, 4.0]))
      .then(pl.lit("Group 4: (3$, 4$)"))
    .alias("price_category")
)

#### 2.4.4. Final variable analysis

In [13]:
# ------------------------------------------------------------
# Build zone-level busyness = transactions / spaces
# ------------------------------------------------------------
tx_by_zone = (
    tx
    .group_by("zone_group_norm")
    .agg(
        pl.col("n_transaction").sum().alias("transactions")
    )
)

busyness = (
    tx_by_zone
    .join(spaces, on="zone_group_norm", how="inner")
    .with_columns(
        (pl.col("transactions") / pl.col("spaces")).alias("transactions_per_space")
    )
)


# ------------------------------------------------------------
# Combine zone-level price and busyness into one table
# This is the core table for the analysis.
# ------------------------------------------------------------
price_busyness = (
    busyness
    .join(avg_rates, on="zone_group_norm", how="inner")
)

**Price & Busyness Relationship Plots**

In [ ]:
# ------------------------------------------------------------
# Summarize busyness by price category
# Useful for seeing whether more expensive categories are also
# busier on average.
# ------------------------------------------------------------
by_price_category = (
    price_busyness
    .group_by("price_category")
    .agg([
        pl.col("transactions_per_space").mean().alias("mean_busyness"),
        pl.col("transactions_per_space").std().alias("std_busyness"),
        pl.col("spaces").mean().alias("mean_spaces"),
        pl.col("spaces").std().alias("std_spaces"),
        pl.col("zone_group_norm").n_unique().alias("n_zones"),
    ])
    .sort("price_category")
)

# average busyness by price level
avg_tx_space_bars = alt.Chart(by_price_category.to_pandas()).mark_bar().encode(
    x=alt.X("price_category:N", 
            title="Level of Price", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("mean_busyness:Q", title="Average transactions per space"),
    color=alt.Color("mean_busyness:Q"),
    tooltip=[
        "price_category:O",
        alt.Tooltip("mean_busyness:Q", format=".3f"),
        alt.Tooltip("std_busyness:Q", format=".3f"),
        "n_zones:Q",
    ],
).properties(
    title="Figure 2.1 — Mean busyness by price level",
    width=400,
    height=380,
)

# --------------------------------------------------------------------
# Scatter + regression: each point is a zone.
# It shows how busyness (transactions per space) changes with price,
# while the regression line shows the overall trend.
# --------------------------------------------------------------------
avg_tx_space_scatter = alt.Chart(price_busyness.to_pandas()).mark_circle(size=70, opacity=0.6).encode(
    x=alt.X("rate:Q", 
            title="Hourly rate ($/hour)", scale=alt.Scale(domain=[0.5, 4.5]),),
    y=alt.Y("transactions_per_space:Q", title="Average transactions per space"),
    tooltip=[
        "zone_group_norm:N",
        alt.Tooltip("rate:Q", format=".2f"),
        alt.Tooltip("transactions_per_space:Q", format=".3f"),
        "price_category:N",
        alt.Tooltip("spaces:Q", format=",.0f")
    ]
).properties(width=400, height=380, title="Figure 2.2 — Price-busyness scatter with regression trend")

# regression trend line, it shows the overall trend between parking price and busyness
trend = alt.Chart(price_busyness.to_pandas()).transform_regression(
    "rate", "transactions_per_space"
).mark_line().encode(
    x="rate:Q",
    y="transactions_per_space:Q"
)

# boxplot to compare the distribution of busyness
avg_tx_space_box = (
    alt.Chart(price_busyness.to_pandas())
    .mark_boxplot(size=45)
    .encode(
        # x-axis: categorical price groups
        x=alt.X(
            "price_category:N",
            title="Level of Price",
            axis=alt.Axis(labelAngle=0)  # keep labels horizontal
        ),
        # y-axis: zone-level busyness proxy
        y=alt.Y(
            "transactions_per_space:Q",
            title="Transactions per space"
        ),
        # tooltip: show the price group when hovering
        tooltip=[
            alt.Tooltip("price_category:N", title="Price level")
        ]
    )
    .properties(
        title="Figure 2.3 - Distribution of busyness by price level",
        width=400,
        height=380
    )
)


# Display the chart
avg_tx_space_bars | avg_tx_space_scatter + trend

alt.HConcatChart(...)

In [15]:
# Boxplot display
avg_tx_space_box

alt.Chart(...)

In [16]:
# ------------------------------------------------------------
# Distribution of zones across price categories
# This shows how many zones fall into each price level.
# ------------------------------------------------------------
avg_rate_dist = (
    avg_rates
    .group_by("price_category")
    .len()
    .rename({"len": "count"})
    .with_columns(
        # convert counts to percentages for optional reporting
        (pl.col("count") / pl.col("count").sum() * 100).alias("percent")
    )
    .sort("price_category")
)

rate_bars_grouped = (
    alt.Chart(avg_rate_dist.to_pandas())
    .mark_bar()
    .encode(
        # x-axis: grouped hourly price categories
        x=alt.X(
            "price_category:O",
            title="Rate ($/hour)",
            axis=alt.Axis(labelAngle=0)
        ),
        # y-axis: number of zones in each category
        y=alt.Y(
            "count:Q",
            title="Number of zones"
        ),
        # color bars by count for easier visual comparison
        color=alt.Color(
            "count:Q",
            scale=alt.Scale(scheme="purples"),
            title="Number of zones"
        ),
        # tooltip for exact values on hover
        tooltip=[
            alt.Tooltip("price_category:O", title="Rate"),
            alt.Tooltip("count:Q", title="N_zone"),
            alt.Tooltip("percent:Q", format=".1f", title="Percent")
        ],
    )
    .properties(
        title="Figure 2.4 — Distribution of zones and average spaces by price level",
        width=400,
        height=320,
    )
)

# ------------------------------------------------------------
# Average spaces by price category
# This helps check whether more expensive zones also tend to
# have larger or smaller parking supply.
# ------------------------------------------------------------
avg_spaces_bars = (
    alt.Chart(by_price_category.to_pandas())
    .mark_bar()
    .encode(
        # x-axis: price category
        x=alt.X(
            "price_category:N",
            title="Level of Price",
            axis=alt.Axis(labelAngle=0)
        ),
        # y-axis: average number of spaces across zones in that category
        y=alt.Y(
            "mean_spaces:Q",
            title="Average spaces"
        ),
        # color by average spaces for visual emphasis
        color=alt.Color(
            "mean_spaces:Q",
            title="Average spaces"
        ),
        # tooltip with summary statistics
        tooltip=[
            "price_category:O",
            alt.Tooltip("mean_spaces:Q", format=".3f", title="Mean spaces"),
            alt.Tooltip("std_spaces:Q", format=".3f", title="Std spaces"),
            alt.Tooltip("n_zones:Q", title="N_zone"),
        ],
    )
    .properties(
        title="Figure 2.5 — Distribution of spaces within each price level",
        width=400,
        height=320,
    )
)

# Charts display
(
    (rate_bars_grouped | avg_spaces_bars)
    .resolve_scale(color="independent")
    .resolve_axis(y="independent")
)

alt.HConcatChart(...)

In [20]:
# boxplot to compare the distribution of parking supply
box_spaces = (
    alt.Chart(price_busyness.to_pandas())
    .mark_boxplot(size=40)
    .encode(
        # x-axis: categorical price groups
        x=alt.X(
            "price_category:N",
            title="Price level",
            axis=alt.Axis(labelAngle=0),  # keep labels horizontal
        ),
        # y-axis: number of spaces available in each zone
        y=alt.Y(
            "spaces:Q",
            title="Spaces per zone"
        ),
        # tooltip: show the price category when hovering
        tooltip=[
            alt.Tooltip("price_category:N", title="Price level")
        ],
    )
    .properties(
        title="Figure 2.5 — Spaces per zone by price level",
        width=650,
        height=380,
    )
)

# display the chart
box_spaces

alt.Chart(...)

In [18]:
# ------------------------------------------------------------
# Focus on the middle price groups (Groups 2 and 3)
# These are the most interesting for mismatch analysis because
# they have great standard devation.
# ------------------------------------------------------------
target_groups = ["Group 2: (1.5$)", "Group 3: (2$, 2.25$)"]

g23 = price_busyness.filter(
    pl.col("price_category").is_in(target_groups)
)


# ------------------------------------------------------------
# Compute within-group thresholds
# We use the 10th and 90th percentiles of transactions_per_space
# to identify unusually under-busy / over-busy zones relative to
# other zones in the same price band.
# ------------------------------------------------------------
thr = (
    g23
    .group_by("price_category")
    .agg([
        pl.col("transactions_per_space").quantile(0.10).alias("q10"),
        pl.col("transactions_per_space").quantile(0.90).alias("q90"),
        pl.col("transactions_per_space").median().alias("median"),
        pl.col("transactions_per_space").len().alias("n_zones"),
    ])
)


# ------------------------------------------------------------
# Label mismatch zones
# Over-busy  = top 10% busyness within the price group
# Under-busy = bottom 10% busyness within the price group
# ------------------------------------------------------------
mismatch = (
    g23
    .join(thr, on="price_category", how="left")
    .with_columns(
        pl.when(pl.col("transactions_per_space") >= pl.col("q90"))
          .then(pl.lit("Over-busy (top 10%)"))
        .when(pl.col("transactions_per_space") <= pl.col("q10"))
          .then(pl.lit("Under-busy (bottom 10%)"))
        .otherwise(pl.lit("Typical"))
        .alias("mismatch_type")
    )
    .filter(pl.col("mismatch_type") != "Typical")
    .select([
        "price_category",
        "zone_group_norm",
        "rate",
        "transactions_per_space",
        "spaces",
        "mismatch_type",
    ])
    .sort(["price_category", "transactions_per_space"], descending=[False, True])
)


# ------------------------------------------------------------
# Measure how far each mismatch zone is from its group median
# This gives a cleaner way to visualize the severity of mismatch.
# ------------------------------------------------------------
mm_dev = (
    g23
    .join(thr, on="price_category", how="left")
    .with_columns([
        (pl.col("transactions_per_space") - pl.col("median")).alias("dev_from_median"),
        pl.when(pl.col("transactions_per_space") >= pl.col("q90"))
          .then(pl.lit("Over-busy"))
          .when(pl.col("transactions_per_space") <= pl.col("q10"))
          .then(pl.lit("Under-busy"))
          .otherwise(pl.lit(None))
          .alias("mismatch_type"),
    ])
    .filter(pl.col("mismatch_type").is_not_null())
    .select([
        "price_category",
        "zone_group_norm",
        "rate",
        "spaces",
        "transactions_per_space",
        "median",
        "dev_from_median",
        "mismatch_type",
    ])
    .sort(["price_category", "dev_from_median"], descending=[False, True])
)


# ------------------------------------------------------------
# Convert to pandas for Altair plotting
# ------------------------------------------------------------
mm_dev_pd = mm_dev.to_pandas()

g2_label = "Group 2: (1.5$)"
g3_label = "Group 3: (2$, 2.25$)"

g2 = mm_dev_pd[mm_dev_pd["price_category"] == g2_label]
g3 = mm_dev_pd[mm_dev_pd["price_category"] == g3_label]


# ------------------------------------------------------------
# Plot mismatch severity within each price group
# Positive deviation = busier than the group median
# Negative deviation = less busy than the group median
# ------------------------------------------------------------
def mismatch_bar(df, title):
    """
    Create a bar chart showing how far each mismatch zone deviates
    from the median busyness of its price category.

    Parameters
    df : pandas.DataFrame
        Subset of mismatch zones for one price category.
    title : str
        Chart title.

    Returns
    alt.Chart
        Altair bar chart of deviation from group median.
    """
    return (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(
                "dev_from_median:Q",
                title="Deviation from group median (tx/space)"
            ),
            y=alt.Y(
                "zone_group_norm:N",
                sort="-x",
                title="Zone"
            ),
            color=alt.Color(
                "dev_from_median:Q",
                title="Mismatch"
            ),
            tooltip=[
                "zone_group_norm:N",
                alt.Tooltip("rate:Q", format=".2f", title="Rate"),
                alt.Tooltip("transactions_per_space:Q", format=".3f", title="Tx/space"),
                alt.Tooltip("median:Q", format=".3f", title="Group median"),
                alt.Tooltip("dev_from_median:Q", format=".3f", title="Deviation"),
                alt.Tooltip("spaces:Q", format=",.0f", title="Spaces"),
                "mismatch_type:N",
            ],
        )
        .properties(width=330, height=380, title=title)
    )


c1 = mismatch_bar(g2, "Figure 2.6a — Group 2 mismatch zones relative to group median")
c2 = mismatch_bar(g3, "Figure 2.6b — Group 3 mismatch zones relative to group median")

(c1 | c2).resolve_scale(x="shared", color="shared")

alt.HConcatChart(...)

**Top Zones per Group Display**

List top zones by transaction per space for each group

In [19]:
unique_price_category_list = (
    price_busyness.select("price_category").unique().to_series().to_list()
)
unique_price_category_list = sorted(unique_price_category_list)

charts = []
for r in unique_price_category_list:
    tmp_pb = (
        price_busyness
        .filter(pl.col("price_category") == r)
        .sort("transactions_per_space", descending=True)
        .head(7)
        .to_pandas()
    )

    c = alt.Chart(tmp_pb).mark_bar().encode(
        x=alt.X("transactions_per_space:Q", title="Transactions per space"),
        y=alt.Y("zone_group_norm:N", sort="-x", title="Top zones"),
        color=alt.Color("transactions_per_space:Q"),
        tooltip=[
            "zone_group_norm:N",
            alt.Tooltip("transactions_per_space:Q", format=".3f"),
            "transactions:Q",
            "spaces:Q",
        ],
    ).properties(
        title=f"Top zones of {r}",
        width=450,
        height=140,
    )

    charts.append(c)

# stack charts vertically
alt.vconcat(*charts)

alt.VConcatChart(...)

### 2.5. Interpretation

The main takeaway is: 
> **prices are generally related to busyness, but the fit is imperfect, especially in the middle price tiers.**


Overall, the figures suggest a **positive relationship between parking price and busyness**. In **Figure 2.1**, average transactions per space rise clearly from **Group 1** to **Group 3** this indicates that **higher-priced zones are generally busier**, so pricing is at least broadly aligned with demand. That said, the pattern is **not perfectly monotonic**. In **Figure 2.1**, **Group 4 ($3–$4)** is slightly **less busy on average than Group 3 ($2–$2.25)**. However, this does not overturn the overall conclusion, because the scatter in **Figure 2.2** still shows an upward overall trend. 

A more likely explanation is that **Group 4 is structurally different** from the other groups rather than simply "mispriced". This interpretation is supported by **Figure 2.4** and **Figure 2.5**. Group 4 contains **fewer zones**, and those zones tend to have **much larger parking supply**. That makes them less directly comparable to the other price tiers. These may be zones with different parking functions, such as larger lots, longer-duration stays, lease or permit-oriented use, or higher operating and maintenance costs. In that case, the city may not be able to set Group 4 prices purely according to short-term demand, and the observed price level may reflect those structural features.

**Key Insight**

The clearest sign that pricing is **not perfectly aligned** with demand comes from **Figure 2.3**. The boxes for **Group 2** and **Group 3** are very wide, which means that zones within the same price tier have **very different levels of busyness**. So even though price and busyness move together on average, many individual zones inside the same tier behave quite differently. This suggests that the current grouped pricing system captures the broad pattern, but still leaves substantial room for mismatch.

That mismatch becomes more concrete in **Figure 2.6a** and **Figure 2.6b**. Some zones are much **busier than the median** for their price group, while others are much **less busy than the median**. In practical terms, this means some zones are likely **underpriced relative to demand**, while others may be **overpriced**. 

#### Action to recommend

A good next step would be to move toward a **more demand-responsive pricing review**, especially for **Groups 2 and 3**, since those tiers show the widest within-group variation. The city could begin by examining the mismatch zones in **Figure 2.6a** and **Figure 2.6b**: 
- zones that are **too busy for their price level** could be candidates for a moderate price increase
- while zones that are **too quiet for their price level** could be candidates for a reduction

At the same time, **Group 4 should probably be treated separately**. Because those zones appear to be fewer, larger, and potentially serving different parking purposes, their pricing may reflect more than just observed transaction demand. Any pricing adjustment there should consider operational cost, long-stay behavior, lease use, and local context.